In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from aqs import collect_openaq as O, collect_trends as T
from aqs.config import CITIES, RAW_AQ, RAW_TRENDS

In [2]:
O.collect()

bangkok: data\raw\openaq\bangkok_raw.csv exists, skipping (use --force)
krakow: data\raw\openaq\krakow_raw.csv exists, skipping (use --force)
melbourne: data\raw\openaq\melbourne_raw.csv exists, skipping (use --force)
lahore: data\raw\openaq\lahore_raw.csv exists, skipping (use --force)


In [3]:
failed = T.collect()
print(f"{len(T.missing_series(list(CITIES)))} search series missing, {len(failed)} failed in this run")

0 search series missing, 0 failed in this run


In [4]:
sensors = pd.concat([pd.read_csv(RAW_AQ / f"{city}_sensors.csv") for city in CITIES])
counts = sensors.groupby("city", sort=False).agg(
    candidate_sensors=("sensor_id", "size"), sensors_with_data=("n_days", lambda n: int((n > 0).sum())),
    reference_monitors=("is_monitor", "sum"), sensor_days=("n_days", "sum"))
counts["search_series_planned"] = [len(T.series_plan(city)) for city in counts.index]
counts["search_series_on_disk"] = [len(list(RAW_TRENDS.glob(f"{city}__*.csv"))) for city in counts.index]
for city, row in counts.iterrows():
    print(f"{city}: {row['candidate_sensors']} sensors ({row['sensors_with_data']} with data, "
          f"{row['sensor_days']} sensor-days), {row['search_series_on_disk']} of "
          f"{row['search_series_planned']} search series")
counts

bangkok: 182 sensors (117 with data, 76669 sensor-days), 20 of 20 search series
krakow: 5 sensors (4 with data, 2012 sensor-days), 19 of 19 search series
melbourne: 25 sensors (25 with data, 8939 sensor-days), 12 of 12 search series
lahore: 2 sensors (2 with data, 1437 sensor-days), 11 of 11 search series


,candidate_sensors,sensors_with_data,reference_monitors,sensor_days,search_series_planned,search_series_on_disk
city,,,,,,
bangkok,182,117,154,76669,20,20
krakow,5,4,3,2012,19,19
melbourne,25,25,21,8939,12,12
lahore,2,2,1,1437,11,11


In [5]:
plan = pd.DataFrame([(city, geo, key, keyword) for city in CITIES for geo, key, keyword in T.series_plan(city)],
                    columns=["city", "geo", "term", "keyword"])
plan.groupby(["city", "geo"], sort=False).size().unstack(fill_value=0)

geo,TH-10,TH,PL-MA,PL,AU-VIC,AU,PK-PB,PK
city,,,,,,,,
bangkok,8,12,0,0,0,0,0,0
krakow,0,0,7,12,0,0,0,0
melbourne,0,0,0,0,4,8,0,0
lahore,0,0,0,0,0,0,3,8
